# 🎯 JobFit Agent - Demostración Completa

**Agente Inteligente para Auditoría de Ofertas y Adaptación de CV**

Este notebook demuestra todas las funcionalidades del JobFit Agent:
- 🔍 Scraping y auditoría de ofertas
- 📊 Cálculo de scores de realismo
- 🎯 Matching semántico CV-Oferta
- 📄 Adaptación inteligente de CV
- 📈 Análisis de resultados

---

## 📋 Setup Inicial

In [ ]:
# Imports y configuración
import sys
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Añadir src al path
sys.path.append('../src')
sys.path.append('..')

# Configurar directorios
os.chdir('..')
print(f"📁 Directorio actual: {os.getcwd()}")

# Importar nuestros módulos
from src.scraper.job_scraper import JobScraper
from src.auditor.realism_scorer import RealismScorer
from src.extractor.job_parser import JobParser
from src.extractor.cv_parser import CVParser
from src.matcher.semantic_matcher import SemanticMatcher
from src.generator.cv_adapter import CVAdapter

print("✅ Módulos importados correctamente")

In [ ]:
# Verificar dependencias
try:
    import gradio as gr
    import sentence_transformers
    from docx import Document
    import requests
    from bs4 import BeautifulSoup
    print("✅ Todas las dependencias disponibles")
except ImportError as e:
    print(f"❌ Dependencia faltante: {e}")
    print("💡 Ejecuta: pip install -r requirements.txt")

## 🧪 Ejemplo 1: Auditoría de Oferta de Trabajo

Vamos a analizar una oferta de trabajo real para calcular su score de realismo.

In [ ]:
# Ejemplo de oferta de trabajo (texto simulado)
sample_job_offer = """
Desarrollador Full Stack Senior - Madrid

Buscamos un Desarrollador Full Stack Senior con 8+ años de experiencia para unirse a nuestro equipo de tecnología.

Requisitos imprescindibles:
• 8+ años de experiencia en desarrollo web
• Dominio experto en Python, JavaScript, React, Angular, Vue.js
• Experiencia con Node.js, Django, Flask, Spring Boot
• Conocimientos avanzados en AWS, Azure, Docker, Kubernetes
• Experiencia en SQL, MongoDB, PostgreSQL, Redis
• Dominio de Git, Jenkins, CI/CD
• Experiencia en desarrollo móvil (iOS, Android, React Native)
• Conocimientos en Machine Learning y Data Science
• Inglés nativo y alemán conversacional

Requisitos deseables:
• Master en Informática
• Certificaciones AWS y Azure
• Experiencia liderando equipos
• Conocimientos en blockchain

Ofrecemos:
• Salario: 35.000 - 40.000€
• Trabajo 100% presencial en Madrid
• Horario de 9:00 a 21:00, sábados incluidos

¡Únete a nuestro equipo dinámico!
"""

print("📄 Oferta de ejemplo cargada")
print(f"📊 Longitud: {len(sample_job_offer)} caracteres")

In [ ]:
# Inicializar componentes
job_parser = JobParser()
realism_scorer = RealismScorer()

print("🤖 Componentes inicializados")

In [ ]:
# Extraer datos estructurados de la oferta
job_data = job_parser.extract_job_data(sample_job_offer)

print("📋 DATOS EXTRAÍDOS:")
print("=" * 50)
for key, value in job_data.items():
    if isinstance(value, list):
        print(f"📌 {key}: {len(value)} items")
        for item in value[:3]:  # Mostrar primeros 3
            print(f"   • {item}")
        if len(value) > 3:
            print(f"   ... y {len(value)-3} más")
    else:
        print(f"📌 {key}: {value}")
    print()

In [ ]:
# Calcular score de realismo
audit_result = realism_scorer.calculate_realism_score(job_data)

print("📊 AUDITORÍA DE REALISMO:")
print("=" * 50)
print(f"🎯 Score General: {audit_result['realism_score']}/100")

# Mostrar scores por categoría
categories = audit_result.get('categories', {})
print("\n📈 Scores por Categoría:")
for category, score in categories.items():
    if score is not None:
        emoji = "🟢" if score >= 80 else "🟡" if score >= 60 else "🟠" if score >= 40 else "🔴"
        print(f"   {emoji} {category.replace('_', ' ').title()}: {score}/100")

# Mostrar señales detectadas
print("\n🚨 Señales Detectadas:")
for signal in audit_result['signals']:
    emoji = "⚠️" if signal['type'] == 'warning' else "❌" if signal['type'] == 'error' else "ℹ️"
    print(f"   {emoji} {signal['description']} (Impacto: {signal['impact']}/10)")

print(f"\n💭 Razonamiento: {audit_result['reasoning']}")

## 📄 Ejemplo 2: Procesamiento de CV

Vamos a crear un CV de ejemplo y procesarlo.

In [ ]:
# Crear CV de ejemplo
sample_cv_text = """
Juan Pérez García
juan.perez@email.com | +34 666 123 456
Madrid, España

EXPERIENCIA PROFESIONAL

Desarrollador Full Stack | TechCorp S.L. | 2020-2024
• Desarrollo de aplicaciones web con React y Node.js
• Implementación de APIs REST con Python y Django
• Gestión de bases de datos PostgreSQL y MongoDB
• Despliegue en AWS usando Docker
• Colaboración en equipo usando Git y metodologías ágiles

Desarrollador Junior | StartupXYZ | 2018-2020
• Desarrollo frontend con JavaScript y HTML/CSS
• Mantenimiento de código legacy en PHP
• Participación en code reviews y testing
• Aprendizaje de frameworks modernos

Prácticas en Desarrollo | Universidad Complutense | 2017-2018
• Proyecto final: aplicación web de gestión académica
• Tecnologías: Java, Spring, MySQL
• Trabajo en equipo y documentación técnica

EDUCACIÓN

Grado en Ingeniería Informática | Universidad Complutense Madrid | 2014-2018
• Especialización en Ingeniería de Software
• Proyecto final: Sistema de recomendación con Machine Learning
• Promedio: 7.5/10

HABILIDADES TÉCNICAS

• Lenguajes: Python, JavaScript, Java, HTML, CSS, SQL
• Frameworks: React, Django, Node.js, Spring
• Bases de datos: PostgreSQL, MongoDB, MySQL
• Cloud: AWS (EC2, S3, RDS)
• Herramientas: Git, Docker, Jenkins
• Metodologías: Scrum, Kanban

PROYECTOS DESTACADOS

E-commerce Platform | 2023
• Plataforma completa de comercio electrónico
• Frontend en React con Redux
• Backend en Django con API REST
• Pagos integrados con Stripe
• Desplegado en AWS

Task Management App | 2022
• Aplicación de gestión de tareas tipo Trello
• Desarrollada con MERN stack
• Autenticación JWT
• Real-time con Socket.io

IDIOMAS
• Español: Nativo
• Inglés: B2 (Intermedio Alto)
"""

# Guardar CV temporal
cv_path = "data/temp/sample_cv.txt"
Path(cv_path).parent.mkdir(parents=True, exist_ok=True)

with open(cv_path, 'w', encoding='utf-8') as f:
    f.write(sample_cv_text)

print(f"📄 CV de ejemplo guardado en: {cv_path}")
print(f"📊 Longitud: {len(sample_cv_text)} caracteres")

In [ ]:
# Parsear el CV
cv_parser = CVParser()
cv_data = cv_parser.parse_cv(cv_path, "txt")

print("📋 CV PARSEADO:")
print("=" * 50)

# Información personal
print("👤 Información Personal:")
for key, value in cv_data['personal_info'].items():
    print(f"   📌 {key}: {value}")

# Experiencia
print(f"\n💼 Experiencia Profesional ({len(cv_data['experience'])} puestos):")
for i, exp in enumerate(cv_data['experience'][:3]):
    print(f"   {i+1}. {exp.get('title', 'N/A')} en {exp.get('company', 'N/A')}")
    print(f"      📅 {exp.get('period', 'N/A')}")
    if exp.get('description'):
        print(f"      📝 {len(exp['description'])} logros/responsabilidades")
    print()

# Habilidades
print(f"🛠️ Habilidades ({len(cv_data['skills'])} total):")
for skill in cv_data['skills'][:10]:
    print(f"   • {skill}")
if len(cv_data['skills']) > 10:
    print(f"   ... y {len(cv_data['skills'])-10} más")

# Proyectos
print(f"\n🚀 Proyectos ({len(cv_data['projects'])} total):")
for project in cv_data['projects']:
    print(f"   📁 {project.get('name', 'Proyecto')}")
    if project.get('description'):
        print(f"      📝 {len(project['description'])} detalles")

## 🎯 Ejemplo 3: Matching Semántico

Ahora vamos a hacer el matching entre la oferta y el CV.

In [ ]:
# Inicializar matcher semántico
semantic_matcher = SemanticMatcher()

print("🧠 Cargando modelo de embeddings...")
print("💡 Esto puede tardar unos minutos la primera vez")

# Preparar datos para matching
requirements = job_data.get('must_have', []) + job_data.get('nice_to_have', [])
print(f"📋 Total requisitos: {len(requirements)}")

# Preparar secciones del CV
cv_sections = {
    'experience': ' '.join([f"{exp.get('title', '')} {exp.get('company', '')} {' '.join(exp.get('description', []))}" for exp in cv_data.get('experience', [])]),
    'skills': ' '.join(cv_data.get('skills', [])),
    'projects': ' '.join([f"{proj.get('name', '')} {' '.join(proj.get('description', []))}" for proj in cv_data.get('projects', [])]),
    'education': ' '.join([edu.get('degree', '') for edu in cv_data.get('education', [])])
}

print("📊 Secciones del CV preparadas:")
for section, content in cv_sections.items():
    print(f"   📌 {section}: {len(content)} caracteres")

In [ ]:
# Realizar matching semántico
print("🎯 Realizando matching semántico...")
matching_results = semantic_matcher.match_requirements_to_cv(requirements, cv_sections)

print("\n📊 RESULTADOS DEL MATCHING:")
print("=" * 60)

coverage = matching_results.get('coverage', 0)
matches = matching_results.get('matches', [])
gaps = matching_results.get('gaps', [])

print(f"🎯 Cobertura General: {coverage*100:.1f}%")
print(f"✅ Requisitos Cubiertos: {len(matches)}")
print(f"❌ Requisitos Faltantes: {len(gaps)}")

# Análisis de cobertura
if coverage >= 0.8:
    print("🟢 Excelente encaje - Candidato muy adecuado")
elif coverage >= 0.6:
    print("🟡 Buen encaje - Candidato adecuado con algunas áreas de mejora")
elif coverage >= 0.4:
    print("🟠 Encaje parcial - Candidato con potencial pero necesita desarrollo")
else:
    print("🔴 Encaje bajo - Candidato no recomendado para esta posición")

In [ ]:
# Mostrar matches detallados
print("\n✅ REQUISITOS CUBIERTOS:")
print("=" * 50)

# Ordenar por confianza
sorted_matches = sorted(matches, key=lambda x: x.get('confidence', 0), reverse=True)

for i, match in enumerate(sorted_matches[:10], 1):  # Top 10
    confidence = match.get('confidence', 0)
    match_type = match.get('match_type', 'unknown')
    
    # Emojis según confianza
    if confidence >= 0.9:
        emoji = "🔥"
    elif confidence >= 0.7:
        emoji = "✅"
    else:
        emoji = "⚡"
    
    type_emoji = "🎯" if match_type == 'exact' else "🔍"
    
    print(f"{i:2}. {emoji} {type_emoji} **{match['requirement']}**")
    print(f"    💪 Confianza: {confidence:.1%}")
    print(f"    📝 Evidencia: {match['evidence'][:100]}...")
    print(f"    📍 Sección: {match.get('section', 'N/A')}")
    print()

if len(sorted_matches) > 10:
    print(f"... y {len(sorted_matches)-10} coincidencias más")

In [ ]:
# Mostrar gaps (requisitos faltantes)
print("\n❌ REQUISITOS FALTANTES:")
print("=" * 50)

if gaps:
    for i, gap in enumerate(gaps[:8], 1):  # Top 8 gaps
        print(f"{i:2}. ⭕ {gap}")
    
    if len(gaps) > 8:
        print(f"... y {len(gaps)-8} requisitos más")
    
    print("\n💡 Recomendaciones:")
    print("   • Considera desarrollar las competencias faltantes")
    print("   • Enfócate en los 2-3 requisitos más importantes")
    print("   • Destaca las competencias transferibles que tienes")
else:
    print("🎉 ¡Excelente! Cubres todos los requisitos principales")

## 📄 Ejemplo 4: Adaptación de CV

Ahora vamos a adaptar el CV para que encaje mejor con la oferta.

In [ ]:
# Inicializar adaptador de CV
cv_adapter = CVAdapter()

print("📝 Adaptando CV a la oferta...")

# Adaptar CV
adapted_cv = cv_adapter.adapt_cv(cv_data, job_data, matching_results)

print("✅ CV adaptado exitosamente")
print("\n📋 ADAPTACIONES REALIZADAS:")
print("=" * 50)

# Notas de adaptación
notes = adapted_cv.get('adaptation_notes', {})
print(f"📊 Cobertura: {notes.get('coverage_percentage', 0)}%")
print(f"✅ Requisitos cubiertos: {notes.get('matched_requirements', 0)}")
print(f"❌ Requisitos faltantes: {notes.get('gap_requirements', 0)}")
print(f"🎯 Estrategia: {notes.get('adaptation_strategy', 'N/A')}")

print("\n💡 Recomendaciones:")
for rec in notes.get('recommendations', []):
    print(f"   • {rec}")

In [ ]:
# Mostrar resumen adaptado
print("\n📝 NUEVO RESUMEN PROFESIONAL:")
print("=" * 50)
print(adapted_cv.get('summary', 'No generado'))

print("\n🆚 COMPARACIÓN CON ORIGINAL:")
print("=" * 50)
print("ANTES: Resumen genérico o inexistente")
print(f"DESPUÉS: Resumen dirigido a la oferta específica")
print(f"✨ Mejora: Enfoque específico en la posición de {job_data.get('title', 'Desarrollador')}")

In [ ]:
# Mostrar experiencia reordenada
print("\n💼 EXPERIENCIA REORDENADA POR RELEVANCIA:")
print("=" * 50)

original_exp = cv_data.get('experience', [])
adapted_exp = adapted_cv.get('experience', [])

print("📊 Comparación de orden:")
print("\nORDEN ORIGINAL:")
for i, exp in enumerate(original_exp, 1):
    print(f"{i}. {exp.get('title', 'N/A')} - {exp.get('company', 'N/A')}")

print("\nORDEN ADAPTADO (por relevancia):")
for i, exp in enumerate(adapted_exp, 1):
    print(f"{i}. {exp.get('title', 'N/A')} - {exp.get('company', 'N/A')}")
    # Mostrar si cambió de posición
    original_pos = next((j for j, orig in enumerate(original_exp) if orig.get('title') == exp.get('title')), -1)
    if original_pos != i-1:
        print(f"   📈 Movido desde posición {original_pos+1} (mayor relevancia)")

In [ ]:
# Mostrar skills reorganizadas
skills = adapted_cv.get('skills', {})

print("\n🛠️ HABILIDADES REORGANIZADAS:")
print("=" * 50)

if skills.get('highlighted'):
    print("⭐ Habilidades Destacadas (con match en la oferta):")
    for skill in skills['highlighted'][:8]:
        print(f"   🔥 {skill}")

print("\n🔧 Habilidades Técnicas (priorizadas):")
for skill in skills.get('technical', [])[:10]:
    highlight = "⭐" if skill in skills.get('highlighted', []) else "•"
    print(f"   {highlight} {skill}")

if skills.get('other'):
    print("\n💼 Otras Competencias:")
    for skill in skills.get('other', [])[:5]:
        print(f"   • {skill}")

## 📊 Ejemplo 5: Exportación y Análisis Final

In [ ]:
# Exportar CV adaptado
output_path = "exports/cv_adaptado_demo.docx"
Path(output_path).parent.mkdir(parents=True, exist_ok=True)

exported_path = cv_adapter.export_to_docx(adapted_cv, output_path)

print(f"📄 CV adaptado exportado: {exported_path}")
print(f"📁 Ubicación: {Path(exported_path).absolute()}")

# Verificar que el archivo existe
if Path(exported_path).exists():
    file_size = Path(exported_path).stat().st_size / 1024  # KB
    print(f"✅ Archivo creado exitosamente ({file_size:.1f} KB)")
else:
    print("❌ Error creando archivo")

In [ ]:
# Generar reporte completo
complete_report = {
    "job_analysis": {
        "title": job_data.get('title'),
        "realism_score": audit_result['realism_score'],
        "main_issues": [s['description'] for s in audit_result['signals'] if s['type'] == 'error'],
        "total_requirements": len(requirements)
    },
    "cv_analysis": {
        "total_experience_years": len(cv_data.get('experience', [])),
        "total_skills": len(cv_data.get('skills', [])),
        "total_projects": len(cv_data.get('projects', []))
    },
    "matching_analysis": {
        "coverage_percentage": round(coverage * 100, 1),
        "requirements_covered": len(matches),
        "requirements_missing": len(gaps),
        "top_strengths": [m['requirement'] for m in sorted_matches[:3]],
        "main_gaps": gaps[:3]
    },
    "adaptation_summary": {
        "strategy": notes.get('adaptation_strategy'),
        "key_changes": [
            "Resumen personalizado para la oferta",
            "Experiencia reordenada por relevancia", 
            "Habilidades priorizadas según matching",
            "Proyectos destacados por relevancia"
        ],
        "recommendations": notes.get('recommendations', [])
    }
}

# Guardar reporte
report_path = "exports/complete_report.json"
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(complete_report, f, indent=2, ensure_ascii=False)

print(f"📊 Reporte completo guardado: {report_path}")

In [ ]:
# Mostrar resumen final
print("\n" + "=" * 60)
print("🎯 RESUMEN FINAL DEL ANÁLISIS")
print("=" * 60)

print(f"\n📋 Oferta Analizada:")
print(f"   🎯 Puesto: {job_data.get('title', 'N/A')}")
print(f"   📊 Score Realismo: {audit_result['realism_score']}/100")
print(f"   📍 Ubicación: {job_data.get('location', 'N/A')}")
print(f"   💰 Salario: {job_data.get('salary_range', 'No especificado')}")

print(f"\n👤 Candidato:")
print(f"   📛 Nombre: {cv_data['personal_info'].get('name', 'N/A')}")
print(f"   💼 Experiencia: {len(cv_data.get('experience', []))} puestos")
print(f"   🛠️ Habilidades: {len(cv_data.get('skills', []))} skills")
print(f"   🚀 Proyectos: {len(cv_data.get('projects', []))} proyectos")

print(f"\n🎯 Matching:")
coverage_emoji = "🟢" if coverage >= 0.8 else "🟡" if coverage >= 0.6 else "🟠" if coverage >= 0.4 else "🔴"
print(f"   {coverage_emoji} Cobertura: {coverage*100:.1f}%")
print(f"   ✅ Requisitos cubiertos: {len(matches)}/{len(requirements)}")
print(f"   ❌ Gaps principales: {len(gaps)}")

print(f"\n📄 Resultado:")
print(f"   📝 CV adaptado generado")
print(f"   📊 Reporte completo creado")
print(f"   💾 Archivos exportados a: exports/")

# Recomendación final
if coverage >= 0.7:
    print(f"\n🚀 RECOMENDACIÓN: Candidato MUY ADECUADO para la posición")
    print(f"   💡 Proceder con la aplicación usando el CV adaptado")
AMIGO
first
OTO:
    print(f"\n⚡ RECOMENDACIÓN: Candidato con POTENCIAL")
    print(f"   💡 Desarrollar competencias faltantes antes de aplicar")
else:
    print(f"\n⚠️ RECOMENDACIÓN: Posición NO RECOMENDADA")
    print(f"   💡 Buscar posiciones más alineadas con el perfil actual")

print("\n" + "=" * 60)
print("✅ DEMOSTRACIÓN COMPLETADA")
print("=" * 60)

## 📈 Ejemplo 6: Análisis Comparativo

Vamos a crear un DataFrame para analizar los resultados.

In [ ]:
# Crear DataFrame con análisis de matching
matching_df = pd.DataFrame([
    {
        'requirement': match['requirement'],
        'confidence': match['confidence'],
        'match_type': match['match_type'],
        'section': match.get('section', 'unknown'),
        'covered': True
    }
    for match in matches
] + [
    {
        'requirement': gap,
        'confidence': 0.0,
        'match_type': 'none',
        'section': 'none',
        'covered': False
    }
    for gap in gaps
])

print("📊 ANÁLISIS ESTADÍSTICO:")
print("=" * 50)

# Estadísticas generales
total_reqs = len(matching_df)
covered_reqs = len(matching_df[matching_df['covered'] == True])
avg_confidence = matching_df[matching_df['covered'] == True]['confidence'].mean() if covered_reqs > 0 else 0

print(f"📋 Total de requisitos: {total_reqs}")
print(f"✅ Requisitos cubiertos: {covered_reqs} ({covered_reqs/total_reqs*100:.1f}%)")
print(f"❌ Requisitos faltantes: {total_reqs - covered_reqs} ({(total_reqs-covered_reqs)/total_reqs*100:.1f}%)")
print(f"💪 Confianza promedio: {avg_confidence:.1%}")

# Análisis por tipo de match
print("\n🔍 Distribución por tipo de match:")
match_type_counts = matching_df[matching_df['covered'] == True]['match_type'].value_counts()
for match_type, count in match_type_counts.items():
    emoji = "🎯" if match_type == 'exact' else "🔍" if match_type == 'semantic' else "❓"
    print(f"   {emoji} {match_type}: {count} ({count/covered_reqs*100:.1f}%)")

# Top 5 mejores matches
print("\n🏆 Top 5 Mejores Matches:")
top_matches = matching_df[matching_df['covered'] == True].nlargest(5, 'confidence')
for i, (_, row) in enumerate(top_matches.iterrows(), 1):
    print(f"   {i}. {row['requirement'][:50]}... ({row['confidence']:.1%})")

# Mostrar DataFrame
print(f"\n📊 Vista de datos (primeras 10 filas):")
display_df = matching_df.head(10)[['requirement', 'confidence', 'match_type', 'covered']].copy()
display_df['requirement'] = display_df['requirement'].str[:40] + '...'
display_df['confidence'] = display_df['confidence'].apply(lambda x: f"{x:.1%}")
print(display_df.to_string(index=False))

## 🎮 Ejemplo 7: Modo Interactivo

Vamos a crear una función para probar con diferentes ofertas.

In [ ]:
def analyze_job_offer_interactive(job_text, show_details=True):
    """
    Función interactiva para analizar ofertas de trabajo
    """
    print("🚀 Iniciando análisis...")
    
    # Parsear oferta
    job_data = job_parser.extract_job_data(job_text)
    audit_result = realism_scorer.calculate_realism_score(job_data)
    
    # Mostrar resumen rápido
    score = audit_result['realism_score']
    score_emoji = "🟢" if score >= 80 else "🟡" if score >= 60 else "🟠" if score >= 40 else "🔴"
    
    print(f"\n{score_emoji} Score de Realismo: {score}/100")
    print(f"🎯 Puesto: {job_data.get('title', 'No identificado')}")
    print(f"📍 Ubicación: {job_data.get('location', 'No especificada')}")
    print(f"💰 Salario: {job_data.get('salary_range', 'No especificado')}")
    
    if show_details:
        print(f"\n📋 Requisitos encontrados:")
        must_have = job_data.get('must_have', [])
        nice_to_have = job_data.get('nice_to_have', [])
        
        print(f"   ❗ Obligatorios: {len(must_have)}")
        for req in must_have[:3]:
            print(f"      • {req}")
        
        print(f"   💡 Deseables: {len(nice_to_have)}")
        for req in nice_to_have[:3]:
            print(f"      • {req}")
        
        if audit_result['signals']:
            print(f"\n🚨 Principales problemas:")
            for signal in audit_result['signals'][:3]:
                emoji = "❌" if signal['type'] == 'error' else "⚠️"
                print(f"   {emoji} {signal['description']}")
    
    return job_data, audit_result

print("✅ Función interactiva definida")
print("💡 Usa: analyze_job_offer_interactive(texto_de_oferta)")

In [ ]:
# Ejemplo de uso con una oferta más realista
realistic_job_offer = """
Desarrollador Python Backend - Madrid (Híbrido)

En TechStartup buscamos un Desarrollador Python Backend para nuestro equipo de producto.

Requisitos:
• 2-4 años de experiencia en Python
• Experiencia con Django o Flask
• Conocimientos de bases de datos SQL (PostgreSQL)
• Experiencia con Git y desarrollo colaborativo
• Inglés nivel intermedio

Se valorará:
• Experiencia con AWS o Docker
• Conocimientos de testing (pytest)
• Experiencia con APIs REST
• Metodologías ágiles

Ofrecemos:
• Salario: 35.000 - 45.000€
• Modalidad híbrida (2-3 días presencial)
• Formación continua
• Ambiente joven y dinámico
"""

print("🧪 Probando con oferta más realista:")
print("=" * 50)

realistic_job_data, realistic_audit = analyze_job_offer_interactive(realistic_job_offer)

print(f"\n📊 Comparación de scores:")
print(f"   🔴 Oferta original: {audit_result['realism_score']}/100")
print(f"   🟢 Oferta realista: {realistic_audit['realism_score']}/100")
print(f"   📈 Mejora: +{realistic_audit['realism_score'] - audit_result['realism_score']} puntos")

## 🏁 Conclusiones y Próximos Pasos

### ✅ Lo que hemos demostrado:

1. **Auditoría de Ofertas**: Cálculo automático de scores de realismo
2. **Extracción Estructurada**: Conversión de texto libre a JSON
3. **Procesamiento de CV**: Parsing automático de diferentes formatos
4. **Matching Semántico**: Comparación inteligente usando embeddings
5. **Adaptación de CV**: Reordenación sin falsificar información
6. **Exportación**: Generación de documentos DOCX

### 🚀 Cómo usar el sistema:

1. **Interfaz Web**: `python main.py` → http://localhost:7860
2. **Línea de Comandos**: `python main.py --cli --job-url URL --cv-path CV.pdf`
3. **Programáticamente**: Usar los módulos como en este notebook

### 🔧 Posibles mejoras:

- 🌐 Scraping de más portales de empleo
- 🧠 Modelos de embeddings más especializados
- 📊 Dashboard con métricas avanzadas
- 🔌 Integración con LinkedIn/GitHub
- 🌍 Soporte multiidioma

### 💡 Tips de uso:

- Usa CVs bien estructurados para mejores resultados
- Configura API keys de OpenAI para mayor precisión
- Revisa siempre el CV adaptado antes de enviarlo
- Usa el score de realismo para evaluar ofertas

---

**🎉 ¡Gracias por probar JobFit Agent!**

Para más información:
- 📖 README.md
- 🐛 GitHub Issues
- 💬 Discusiones en GitHub